**Note: This should run on a A100 or GPU for faster compute of the reranking** --> please refer to project ReadME for details to run the experiments

to run experiment
- import: generations and gold_answers from the specific k from github (the biggest k)
- this will be used to rerank and generate the new answer
- create a folder in the vs code project for the new results --> import the results and compute the metrics

In [ ]:
!pip install -q sentence-transformers requests tqdm

In [ ]:
# Check GPU availability
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import os
from getpass import getpass

# Get OpenAI API key
os.environ["OPENAI_API_KEY"] = ""

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION - Change these for each experiment
# ═══════════════════════════════════════════════════════════════════════════════
CONFIG = {
    # Input: your large baseline run (e.g., k=100)
    "input_run": "baseline_k50",  # ← Same baseline for ALL experiments

    # Pipeline: baseline_k100 → fetch_k → rerank → top_k
    "fetch_k": 50,  # ← Initial fetch: how many chunks to consider for reranking
    "top_k": 10,     # ← Final: how many chunks to keep after reranking

    # Reranker model (GPU-accelerated)
    "reranker_model": "BAAI/bge-reranker-v2-m3",

    # OpenAI generation model
    "generation_model": "gpt-4o",
}
# ═══════════════════════════════════════════════════════════════════════════════

# Validation
assert CONFIG["top_k"] <= CONFIG["fetch_k"], f"top_k ({CONFIG['top_k']}) must be <= fetch_k ({CONFIG['fetch_k']})"

print("✓ OpenAI API key set")
print("\n📋 Configuration:")
print(f"   Pipeline: {CONFIG['input_run']} → fetch {CONFIG['fetch_k']} → rerank → top {CONFIG['top_k']}")
for k, v in CONFIG.items():
    if "key" not in k.lower():
        print(f"   {k}: {v}")

In [ ]:
from google.colab import files
import os

# Create input directory
input_dir = f"runs/{CONFIG['input_run']}"
os.makedirs(input_dir, exist_ok=True)

print(f"📂 Upload files to: {input_dir}/")
print("   Please upload: generations.jsonl and gold_answers.json")

# Upload files

# Verify
print(f"\n📁 Contents of {input_dir}/:")
for f in os.listdir(input_dir):
    print(f"   - {f}")

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
from sentence_transformers import CrossEncoder
import time

print(f"Loading reranker: {CONFIG['reranker_model']}...")
start = time.time()

reranker = CrossEncoder(
    CONFIG["reranker_model"],
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print(f"✓ Model loaded in {time.time() - start:.1f}s")
print(f"  Device: {reranker.model.device}")

In [ ]:
import json
from datetime import datetime
from typing import List, Dict, Any
from openai import OpenAI


def load_generations(filepath: str) -> List[Dict]:
    """Load generations from JSONL file."""
    records = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records


def load_gold_answers(filepath: str) -> Dict[str, str]:
    """Load gold answers from JSON file."""
    with open(filepath, "r", encoding="utf-8") as f:
        return json.load(f)


def rerank_chunks(reranker, query: str, chunks: List[Dict], top_k: int) -> List[Dict]:
    """
    Rerank chunks using CrossEncoder (GPU-accelerated).

    Args:
        reranker: CrossEncoder model
        query: The question
        chunks: List of chunk dicts with 'text' field
        top_k: Number of top chunks to return after reranking

    Returns:
        Top-k chunks sorted by rerank score
    """
    if not chunks:
        return []

    # Create query-chunk pairs
    pairs = [(query, c.get("text", c.get("content", str(c)))) for c in chunks]

    # Get scores (GPU accelerated)
    scores = reranker.predict(pairs, show_progress_bar=False)

    # Sort by score descending
    scored_chunks = sorted(zip(chunks, scores), key=lambda x: x[1], reverse=True)

    # Return top_k with updated metadata
    result = []
    for new_rank, (chunk, score) in enumerate(scored_chunks[:top_k]):
        result.append({
            **chunk,
            "original_rank": chunk.get("rank", new_rank),
            "rank": new_rank,
            "rerank_score": float(score),
        })

    return result


def generate_answer_openai(
    client: OpenAI,
    question: str,
    chunks: List[Dict],
    model: str = "gpt-4o",
) -> Dict[str, Any]:
    """
    Generate answer using OpenAI API with reranked chunks.
    """
    # Build context from reranked chunks
    context_text = "\n\n".join([
        f"Document {i+1}:\n{c.get('text', c.get('content', str(c)))}"
        for i, c in enumerate(chunks)
    ])



    messages = [
        {
            "role": "system",
            "content":
               "You are a retrieval-augmented question answering system. "
    "You must answer using only the provided context and no prior or external knowledge. "
    "You may combine information from multiple documents in the context to derive the answer. "
    "Do not introduce new facts, assumptions, or interpretations that are not supported by the context. "
    "If the answer cannot be clearly and directly derived from the provided context, "
    "respond exactly with: I don't know (for English questions) or je ne sais pas (for French questions). "
    "Your answers must be concise, factual, and limited to 1–2 sentences."
        },
        {
            "role": "user",
                  "content": (
            f"Question:\n{question}\n\n"
            f"Documents:\n{context_text}"
        ),

        }
    ]
    start_time = time.time()
    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0,
        )
        latency_ms = (time.time() - start_time) * 1000

        return {
            "success": True,
            "answer": response.choices[0].message.content.strip(),
            "latency_ms": latency_ms,
        }
    except Exception as e:
        return {
            "success": False,
            "answer": "",
            "latency_ms": (time.time() - start_time) * 1000,
            "error": str(e),
        }


def save_results(records: List[Dict], gold_answers: Dict[str, str], output_dir: str):
    """Save results in standard format."""
    os.makedirs(output_dir, exist_ok=True)

    # Save generations.jsonl
    with open(f"{output_dir}/generations.jsonl", "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    # Save gold_answers.json
    with open(f"{output_dir}/gold_answers.json", "w", encoding="utf-8") as f:
        json.dump(gold_answers, f, indent=2, ensure_ascii=False)

    return output_dir


print("✓ Helper functions loaded")

In [ ]:
input_dir = f"runs/{CONFIG['input_run']}"

# Load existing data
generations = load_generations(f"{input_dir}/generations.jsonl")
gold_answers = load_gold_answers(f"{input_dir}/gold_answers.json")

# Check how many chunks are in the baseline
sample_chunks = generations[0].get("retrieved_chunks", [])
original_k = len(sample_chunks)

print(f"✓ Loaded {len(generations)} records from {CONFIG['input_run']}")
print(f"  Original k: {original_k} chunks per question")
print(f"  Will rerank to: {CONFIG['top_k']} chunks")
print(f"\nSample question: {generations[0]['question'][:60]}...")

In [ ]:
def deduplicate_chunks(chunks: List[Dict]) -> List[Dict]:
    seen = set()
    unique = []

    for c in chunks:
        text = c.get("text", c.get("content", ""))
        key = " ".join(text.split()).lower()  # normalize whitespace + case

        if key not in seen:
            seen.add(key)
            unique.append(c)

    return unique


In [ ]:
from tqdm.notebook import tqdm

# Initialize OpenAI client
openai_client = OpenAI()

# Initialize tracking
new_records = []
stats = {
    "rerank_latency_ms": 0,
    "generate_latency_ms": 0,
    "success_count": 0,
}

start_time = time.time()

print(f"\n{'='*70}")
print(f"🚀 RERANK + REGENERATE PIPELINE")
print(f"{'='*70}")
print(f"   Input:     {CONFIG['input_run']} ({original_k} chunks)")
print(f"   Step 1:    Rerank → top {CONFIG['top_k']} (GPU: {CONFIG['reranker_model'].split('/')[-1]})")
print(f"   Step 2:    Generate answer with OpenAI ({CONFIG['generation_model']})")
print(f"{'='*70}\n")

for i, record in enumerate(tqdm(generations, desc="Processing")):
    question = record["question"]
    original_chunks = record.get("retrieved_chunks", [])
    original_chunks = deduplicate_chunks(original_chunks)
    # ─────────────────────────────────────────────────────────────────────
    # Step 1: RERANK existing chunks (GPU) → keep top_k
    # ─────────────────────────────────────────────────────────────────────
    rerank_start = time.time()
    reranked_chunks = rerank_chunks(reranker, question, original_chunks, CONFIG["top_k"])
    rerank_latency = (time.time() - rerank_start) * 1000

    # ─────────────────────────────────────────────────────────────────────
    # Step 2: GENERATE new answer using OpenAI with reranked chunks
    # ─────────────────────────────────────────────────────────────────────
    gen_result = generate_answer_openai(
        client=openai_client,
        question=question,
        chunks=reranked_chunks,
        model=CONFIG["generation_model"],
    )
    answer = gen_result["answer"]
    gen_latency = gen_result["latency_ms"]

    # Track stats
    stats["rerank_latency_ms"] += rerank_latency
    stats["generate_latency_ms"] += gen_latency
    if gen_result["success"]:
        stats["success_count"] += 1

    # Debug first question
    if i == 0:
        print(f"\n🔍 First question debug:")
        print(f"   Question: {question[:60]}...")
        print(f"   Answer: {answer[:100]}..." if answer else "   Answer: (empty)")
        print(f"   Chunks: {len(original_chunks)} → Reranked to {len(reranked_chunks)}")
        print(f"   Latency: Rerank {rerank_latency:.0f}ms + Gen {gen_latency:.0f}ms\n")

    # ─────────────────────────────────────────────────────────────────────
    # Create new record
    # ─────────────────────────────────────────────────────────────────────
    new_records.append({
        "question_id": record["question_id"],
        "question": question,
        "generated_answer": answer,
        "retrieved_chunks": reranked_chunks,
        "method_metadata": {
            "input_run": CONFIG["input_run"],
            "original_k": original_k,
            "reranker_model": CONFIG["reranker_model"],
            "generation_model": CONFIG["generation_model"],
            "top_k": CONFIG["top_k"],
            "pipeline": "rerank_regenerate",
            "gpu_reranking": True,
        },
        "latency_ms": rerank_latency + gen_latency,
    })

# ─────────────────────────────────────────────────────────────────────────
# Summary
# ─────────────────────────────────────────────────────────────────────────
total_time = time.time() - start_time
n = len(generations)

print(f"\n{'='*70}")
print(f"✅ REGENERATION COMPLETE")
print(f"{'='*70}")
print(f"   Successful:        {stats['success_count']}/{n}")
print(f"   Avg Rerank (GPU):  {stats['rerank_latency_ms']/n:.0f}ms")
print(f"   Avg Generate:      {stats['generate_latency_ms']/n:.0f}ms")
print(f"   Wall-clock time:   {total_time:.1f}s")
print(f"{'='*70}")

In [ ]:
generations

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
input_dir = f"runs/{CONFIG['input_run']}"

# Load existing data
generations = load_generations(f"{input_dir}/generations.jsonl")
gold_answers = load_gold_answers(f"{input_dir}/gold_answers.json")

# Check how many chunks are in the baseline
sample_chunks = generations[0].get("retrieved_chunks", [])
baseline_k = len(sample_chunks)

# Validation
assert CONFIG["fetch_k"] <= baseline_k, f"fetch_k ({CONFIG['fetch_k']}) must be <= baseline chunks ({baseline_k})"

print(f"✓ Loaded {len(generations)} records from {CONFIG['input_run']}")
print(f"\n📊 Pipeline:")
print(f"   Baseline:     {baseline_k} chunks")
print(f"   → Fetch:      {CONFIG['fetch_k']} chunks (initial fetch)")
print(f"   → Rerank:     GPU reranking")
print(f"   → Top k:      {CONFIG['top_k']} chunks (final)")
print(f"\nSample question: {generations[0]['question'][:60]}...")

In [ ]:
# Generate run ID based on pipeline params
run_id = f"rerank_f{CONFIG['fetch_k']}_k{CONFIG['top_k']}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
output_dir = f"runs/{run_id}"

# Save results
save_results(new_records, gold_answers, output_dir)

# Verification
print(f"📁 Saved to: {output_dir}/")
print(f"   ├── generations.jsonl ({len(new_records)} records)")
print(f"   └── gold_answers.json")
print(f"\n📊 Pipeline: {CONFIG['input_run']} ({baseline_k}) → fetch {CONFIG['fetch_k']} → rerank → top {CONFIG['top_k']}")

# Quick verification
with open(f"{output_dir}/generations.jsonl", "r") as f:
    test_record = json.loads(f.readline())
    print(f"\n✓ Verification: First record has {len(test_record.get('retrieved_chunks', []))} chunks")